## What is Iterative Retrieval in Agentic RAG?

Combined both Iterative And Self reflection

- Definition:
    - Iterative Retrieval is a dynamic strategy where an AI agent doesn't settle for the first batch of retrieved documents. Instead, it evaluates the adequacy of the initial context, and if necessary, it:

- Refines the query,
- Retrieves again,
- Repeats the process until it’s confident enough to answer the original question.

Why Use It?
In standard RAG:

- A single retrieval step is done, and the LLM uses it to answer.
- If the documents were incomplete or irrelevant, the answer may fail.

In Iterative RAG:

- The agent reflects on the retrieved content and the answer it produced.
- If it’s unsure, it can refine its search (like a human researcher would).

In [1]:
import os
from typing import List
from pydantic import BaseModel
from langchain.chat_models import init_chat_model
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_classic.vectorstores import FAISS
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langgraph.graph import StateGraph, END

c:\Users\mani2\Documents\RAG-Mastery\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import os
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv

os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")
llm=init_chat_model("openai:gpt-4o")

In [3]:
# Load And Embed Documents
docs = TextLoader("internal_docs.txt", encoding="utf-8").load()
chunks = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50).split_documents(docs)
vectorstore = FAISS.from_documents(chunks, OpenAIEmbeddings())
retriever = vectorstore.as_retriever()

In [4]:
# Define Agent State
class IterativeRAGState(BaseModel):
    question: str
    refined_question: str = ""
    retrieved_docs: List[Document] = []
    answer: str = ""
    verified: bool = False
    attempts: int = 0


In [5]:
# Retrieve Node
def retrieve_docs(state: IterativeRAGState) -> IterativeRAGState:
    query = state.refined_question or state.question
    docs = retriever.invoke(query)
    return state.model_copy(update={"retrieved_docs": docs})


In [6]:
# Reflect And Verify
def generate_answer(state: IterativeRAGState) -> IterativeRAGState:
    
    context = "\n\n".join(doc.page_content for doc in state.retrieved_docs)
    prompt = f"""Use the following context to answer the question:

Context:
{context}

Question:
{state.question}
"""
    response = llm.invoke(prompt.strip()).content.strip()
    return state.model_copy(update={"answer": response, "attempts": state.attempts + 1})

In [7]:
# Reflect on answer
def reflect_on_answer(state: IterativeRAGState) -> IterativeRAGState:
    
    prompt = f"""
Evaluate whether the answer below is factually sufficient and complete.

Question: {state.question}
Answer: {state.answer}

Respond 'YES' if it's complete, otherwise 'NO' with feedback.
"""
    feedback = llm.invoke(prompt).content.lower()
    verified = "yes" in feedback
    return state.model_copy(update={"verified": verified})


In [8]:
# Refine query
def refine_query(state: IterativeRAGState) -> IterativeRAGState:
    
    prompt = f"""
The answer appears incomplete. Suggest a better version of the query that would help retrieve more relevant context.

Original Question: {state.question}
Current Answer: {state.answer}
"""
    new_query = llm.invoke(prompt).content.strip()
    return state.model_copy(update={"refined_question": new_query})


In [9]:
builder = StateGraph(IterativeRAGState)

builder.add_node("retrieve", retrieve_docs)
builder.add_node("answer", generate_answer)
builder.add_node("reflect", reflect_on_answer)
builder.add_node("refine", refine_query)

builder.set_entry_point("retrieve")
builder.add_edge("retrieve", "answer")
builder.add_edge("answer", "reflect")

builder.add_conditional_edges(
    "reflect",
    lambda s: END if s.verified or s.attempts >= 2 else "refine"
)

builder.add_edge("refine", "retrieve")
builder.add_edge("answer", END)

graph = builder.compile()


In [10]:
query = "agent loops  and transformer-based systems?"

initial_state = IterativeRAGState(question=query)
final = graph.invoke(initial_state)

print("Final Answer:\n", final["answer"])
print("\n Verified:", final["verified"])
print(" Attempts:", final["attempts"])


Final Answer:
 Agent loops refer to systems where a computational agent continuously interacts with an environment, processes information, and takes actions based on the received data. Transformer-based systems, on the other hand, are machine learning models that use the transformer architecture to process and generate sequences of data, commonly used for natural language processing tasks.

In the context provided, transformer variants such as Reformer, EfficientFormer, LLaMA2, Mistral-7B, and Longformer are examples of transformer-based systems that have been optimized for different tasks and environments. These systems can be integrated into agent loops to enhance their functionalities, such as improving natural language understanding, enabling efficient inference on mobile devices, or providing robust document summarization capabilities.

Using transformer-based systems in agent loops can significantly improve the agent's ability to process large amounts of text or other sequential 

In [11]:
final

{'question': 'agent loops  and transformer-based systems?',
 'refined_question': 'A better version of the query might be: "How do agent loops leverage transformer-based systems, and what are the key considerations for optimizing their performance and integration?" \n\nThis revised query aims to directly address both components—agent loops and transformer-based systems—while also seeking insights into their optimization and integration, which are crucial for effective deployment.',
 'retrieved_docs': [Document(id='fcb4c0d4-627e-47c4-8d73-744f05cbcf90', metadata={'source': 'internal_docs.txt'}, page_content='3. Reformer:\n    - Tested for memory efficiency on embedded devices.\n    - LSH attention led to 60% memory reduction.\n    - Integration challenges with standard transformers due to bucket collisions.\n    - Ongoing investigation for training stability and gradient clipping strategies.'),
  Document(id='d07a530f-2054-44f5-97ee-cc48888a6edc', metadata={'source': 'internal_docs.txt'}

![alt text](download.png)